In [ ]:
import torch
print(torch.__version__)  # PyTorch sürümünü gösterir
print(torch.cuda.is_available())  # GPU kullanılabilir mi?
print(torch.cuda.get_device_name(0))  # GPU modelini yazdırır


In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
# VGG16 modelini elle oluşturma
class VGG16(nn.Module):
    def __init__(self, num_classes=3):
        super(VGG16, self).__init__()
        
        # Konvolüsyonel katmanlar (VGG16 mimarisine göre)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        
        self.conv7 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv8 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        
        self.conv9 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv10 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        
        # Tam bağlantılı katmanlar
        self.fc1 = nn.Linear(512 * 7 * 7, 4096)
        self.fc2 = nn.Linear(4096, 4096)
        self.fc3 = nn.Linear(4096, num_classes)  # Çıkış katmanı (3 sınıf)

        # Dropout katmanları
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(nn.ReLU()(self.conv2(nn.ReLU()(self.conv1(x)))))  # Conv1 ve Conv2
        x = self.pool(nn.ReLU()(self.conv4(nn.ReLU()(self.conv3(x)))))  # Conv3 ve Conv4
        x = self.pool(nn.ReLU()(self.conv6(nn.ReLU()(self.conv5(x)))))  # Conv5 ve Conv6
        x = self.pool(nn.ReLU()(self.conv8(nn.ReLU()(self.conv7(x)))))  # Conv7 ve Conv8
        x = self.pool(nn.ReLU()(self.conv10(nn.ReLU()(self.conv9(x)))))  # Conv9 ve Conv10
        
        x = x.view(-1, 512 * 7 * 7)  # Düzleştirme (Flatten)
        x = self.dropout(nn.ReLU()(self.fc1(x)))  # FC1 ve Dropout
        x = self.dropout(nn.ReLU()(self.fc2(x)))  # FC2 ve Dropout
        x = self.fc3(x)  # Çıkış katmanı (sınıf sayısına göre)
        return x

In [ ]:
# Veriyi yükleme ve dönüştürme işlemleri
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Klasör yapınıza göre veri setlerini yükleme
train_dataset = datasets.ImageFolder("C:/Users/bilal/OneDrive/Masaüstü/ai/dataset/train", transform=transform)  # Eğitim verisi yolu
val_dataset = datasets.ImageFolder("C:/Users/bilal/OneDrive/Masaüstü/ai/dataset/val", transform=transform)      # Doğrulama verisi yolu
test_dataset = datasets.ImageFolder("C:/Users/bilal/OneDrive/Masaüstü/ai/dataset/test", transform=transform)    # Test verisi yolu

# DataLoader'ları oluşturma
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Modeli oluştur
model = VGG16(num_classes=3)  # 3 sınıf

# Modeli GPU'ya yükleme (CUDA destekliyse)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

# Kaybı (loss) ve optimizasyonu ayarlama
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
# Modeli eğitme
num_epochs = 15
train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()  # Eğitim moduna geç
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()  # Gradient sıfırlama
        outputs = model(inputs)  # Modelin tahminleri
        loss = criterion(outputs, labels)  # Kaybı hesaplama
        loss.backward()  # Geri yayılım (backpropagation)
        optimizer.step()  # Ağırlıkları güncelleme
        
        running_loss += loss.item()
        
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}')

# Eğitim sürecini görselleştirme
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Loss grafiği
ax[0].plot(train_losses, label='Loss', color='r')
ax[0].set_title('Loss per Epoch')
ax[0].set_xlabel('Epoch')
ax[0].set_ylabel('Loss')
ax[0].legend()

# Accuracy grafiği
ax[1].plot(train_accuracies, label='Accuracy', color='b')
ax[1].set_title('Accuracy per Epoch')
ax[1].set_xlabel('Epoch')
ax[1].set_ylabel('Accuracy')
ax[1].legend()

plt.tight_layout()
plt.show()

# Modeli kaydetme
torch.save(model.state_dict(), 'vgg16_model.pth')
print("Model başarıyla kaydedildi.")
